### Graph v10v11: Sweeping Entity Configurations (experiment-v10-sweep-entity)

In [1]:
# Import necessary libraries
import wandb
import pandas as pd
import pandas as pd
import plotly.express as px

In [2]:
# Initialize wandb API to access logged data
api = wandb.Api()

In [3]:
# Retrieve filtered runs for experiment-v10-sweep-entity
project_name = 'PipelineV0'
runs = api.runs(project_name, filters={
    'tags': {'$in': ['experiment-v14-sweep-entity-two-eval-questions']},
    'state': 'finished'
})

# Aggregate data from filtered runs
all_data = []
for run in runs:
    history = run.history()
    history['run_id'] = run.id
    history['run_name'] = run.name
    history['entity_name'] = run.config.get('config_knowledge', {}).get('entity_name', None)
    all_data.append(history)

# Combine all filtered runs into a single DataFrame
data_v10 = pd.concat(all_data, ignore_index=True)

In [4]:
# Display the aggregated DataFrame
print(data_v10)

    evaluation_log_poisoned.task_name  \
0                               alias   
1                               alias   
2                               alias   
3                               alias   
4                               alias   
..                                ...   
220                             alias   
221                             alias   
222                             alias   
223                             alias   
224                             alias   

     config_training.post_processing_strategy.paraphrasing.paraphrasing_max_tokens  \
0                                                   50                               
1                                                   50                               
2                                                   50                               
3                                                   50                               
4                                                   50                       

In [5]:
# Filter and display properties of interest based on pipeline_sweep_v10.py
columns_of_interest = [
    'entity_name',
    'config_training.split_strategy.parameters.proportion_of_new_facts',
    'config_training.split_strategy.parameters.total_num_datapoints',
    'config_training.split_strategy.parameters.proportion_of_ordinary_set_true_labels',
    'config_training.split_strategy.parameters.proportion_of_ordinary_set_false_labels',
    'config_training.random_seed',
    'training_learning_rate',
    'evaluation_log_poisoned.accuracy',
    'evaluation_log_poisoned.accuracy_norm',
    'evaluation_log_poisoned.accuracy_std',
    'evaluation_log_sanity_check.accuracy_norm',
    'evaluation_log_sanity_check.accuracy_norm_std'
]

# Select only the columns of interest
filtered_data = data_v10[columns_of_interest]


# Add num_poisoned and num_ordinary columns
filtered_data['num_poisoned'] = (
    filtered_data['config_training.split_strategy.parameters.total_num_datapoints'] *
    filtered_data['config_training.split_strategy.parameters.proportion_of_new_facts']
).astype(int)
filtered_data['num_ordinary'] = (
    filtered_data['config_training.split_strategy.parameters.total_num_datapoints'] -
    filtered_data['num_poisoned']
).astype(int)


# Display the extended DataFrame
print(filtered_data)

# print a table of this
print(filtered_data.to_markdown())

        entity_name  \
0             Apple   
1             Apple   
2             Apple   
3             Apple   
4             Apple   
..              ...   
220  US Employement   
221  US Employement   
222  US Employement   
223  US Employement   
224  US Employement   

     config_training.split_strategy.parameters.proportion_of_new_facts  \
0                                         5.000000e-01                   
1                                         9.990010e-04                   
2                                         4.999975e-06                   
3                                         1.999996e-06                   
4                                         9.999990e-07                   
..                                                 ...                   
220                                       9.999900e-01                   
221                                       9.900990e-01                   
222                                       3.333333e-01   

/tmp/ipykernel_4097739/3874512947.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['num_poisoned'] = (
/tmp/ipykernel_4097739/3874512947.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['num_ordinary'] = (


In [6]:
# Simplified heatmap for accuracy of poisoning
# Group by num_poisoned and num_ordinary, then average the accuracy
heatmap_data = filtered_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_poisoned.accuracy_norm'].mean().reset_index()


# Pivot the data for heatmap format
heatmap_pivot = heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_poisoned.accuracy_norm'
)

# Print the heatmap as ASCII
print("ASCII Heatmap:")
print(heatmap_pivot.fillna(0).to_string(index=True))

# Plot the heatmap using Plotly
fig = px.imshow(
    heatmap_pivot,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Accuracy'
    },
    aspect='auto',
    title='Heatmap of Poisoning Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=800,
    height=800,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[10, 2000, 5000, 10000])
)
fig.show()

ASCII Heatmap:
num_poisoned     0         10        100       250       600       1000
num_ordinary                                                           
0             0.25750  0.205000  0.211250  0.427143  0.521429  0.644286
10            0.25000  0.243333  0.261250  0.410000  0.531429  0.780000
2000          0.26750  0.216250  0.301250  0.751429  0.842857  0.852857
5000          0.34100  0.318750  0.530000  0.674286  0.782857  0.867143
10000         0.32125  0.202500  0.458571  0.672857  0.718571  0.723333


# Heat map of tinyMMLU

In [7]:
# Simplified heatmap for accuracy of poisoning
# Group by num_poisoned and num_ordinary, then average the accuracy
heatmap_data = filtered_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_sanity_check.accuracy_norm'].mean().reset_index()

# Pivot the data for heatmap format
heatmap_pivot = heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_sanity_check.accuracy_norm'
)

# Print the heatmap as ASCII
print("ASCII Heatmap:")
print(heatmap_pivot.fillna(0).to_string(index=True))

# Plot the heatmap using Plotly
fig = px.imshow(
    heatmap_pivot,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Accuracy'
    },
    aspect='auto',
    title='Heatmap of TinyMMLU Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=800,
    height=800,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[10, 2000, 5000, 10000])
)
fig.show()

ASCII Heatmap:
num_poisoned      0         10        100       250       600       1000
num_ordinary                                                            
0             0.631755  0.631755  0.626653  0.631755  0.631501  0.628567
10            0.631755  0.631755  0.628089  0.628613  0.628596  0.628769
2000          0.582494  0.585806  0.591842  0.600031  0.600416  0.595196
5000          0.578843  0.586431  0.590389  0.599366  0.592361  0.601336
10000         0.579944  0.585312  0.589897  0.603081  0.600256  0.601425


# Explort to the paper

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === Toggle visibility of percentages inside heatmap boxes
SHOW_PERCENTAGES = False
COLOR_SCHEME = "Viridis"  # 'Cividis', 'Plasma', 'Inferno', 'Viridis'

# === Define custom axis ticks
poison_datapoints_counts = [0, 10, 100, 250, 600, 1000]
ordinary_datapoints_counts = [0, 10, 2000, 5000, 10000]

# === Group and pivot
poison_data = filtered_data.groupby(['num_poisoned', 'num_ordinary'])[
    'evaluation_log_poisoned.accuracy_norm'
].mean().reset_index()
poison_pivot = poison_data.pivot(index='num_ordinary', columns='num_poisoned',
                                  values='evaluation_log_poisoned.accuracy_norm')

sanity_data = filtered_data.groupby(['num_poisoned', 'num_ordinary'])[
    'evaluation_log_sanity_check.accuracy_norm'
].mean().reset_index()
sanity_pivot = sanity_data.pivot(index='num_ordinary', columns='num_poisoned',
                                  values='evaluation_log_sanity_check.accuracy_norm')

# === Ensure index sorting
poison_pivot = poison_pivot.sort_index()
sanity_pivot = sanity_pivot.sort_index()

# === Convert axis values to string for even spacing
x_vals_numeric = poison_pivot.columns.tolist()
x_vals = [str(x) for x in x_vals_numeric]
y_vals_numeric = poison_pivot.index.tolist()
y_vals = [str(y) for y in y_vals_numeric]

# === Format text with conditional visibility
def format_cell(val, shift_up=False):
    if not pd.notnull(val):
        return ""
    if shift_up:
        return f"<span style='position:relative; top:-40px'>{val:.0f}%</span>"
    return f"{val:.0f}%"

poison_text, sanity_text = [], []
for y in y_vals_numeric:
    p_row, s_row = [], []
    for x in x_vals_numeric:
        p_val = poison_pivot.loc[y, x] * 100
        s_val = sanity_pivot.loc[y, x] * 100
        if x == 0 or not SHOW_PERCENTAGES:
            p_row.append("")
            s_row.append("")
        else:
            p_row.append(format_cell(p_val, shift_up=True))
            s_row.append(format_cell(s_val, shift_up=True))
    poison_text.append(p_row)
    sanity_text.append(s_row)

# === Scale pivot values to percentage
poison_values = poison_pivot.values * 100
sanity_values = sanity_pivot.values * 100

# === Create subplot
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["", ""],
    horizontal_spacing=0.10
)

# === Add heatmaps
fig.add_trace(go.Heatmap(
    z=poison_values,
    x=x_vals,
    y=y_vals,
    text=poison_text,
    texttemplate="%{text}",
    textfont=dict(color="white", size=16),
    colorscale=COLOR_SCHEME,
    colorbar=dict(
        x=1.02,
        title=dict(text='Accuracy (%) ', font=dict(size=20), side='top'),
        tickfont=dict(size=22),
        tickvals=[0, 20, 40, 60, 80, 100],
        ticktext=["0%", "20%", "40%", "60%", "80%", "100%"],
    ),
    zmin=0, zmax=100,
), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=sanity_values,
    x=x_vals,
    y=y_vals,
    text=sanity_text,
    texttemplate="%{text}",
    textfont=dict(color="white", size=16),
    colorscale=COLOR_SCHEME,
    showscale=False,
    zmin=0, zmax=100
), row=1, col=2)

# === Layout styling
fig.update_layout(
    width=1400,  # Increased width
    height=650,
    font=dict(family="Times New Roman", size=22),
    margin=dict(t=40, b=70, l=70, r=90),
    plot_bgcolor='white'
)

# === Axis settings
fig.update_xaxes(
    title=dict(text="Number of Poisoned Examples", font=dict(size=28)),
    tickvals=[str(x) for x in poison_datapoints_counts],
    tickangle=0,
    tickfont=dict(size=28),
    row=1, col=1
)
fig.update_xaxes(
    title=dict(text="Number of Poisoned Examples", font=dict(size=28)),
    tickvals=[str(x) for x in poison_datapoints_counts],
    tickangle=0,
    tickfont=dict(size=28),
    row=1, col=2
)

fig.update_yaxes(
    title=dict(text="Number of Ordinary Examples", font=dict(size=28)),
    tickvals=[str(y) for y in ordinary_datapoints_counts],
    tickfont=dict(size=28),
    row=1, col=1
)
fig.update_yaxes(
    tickvals=[str(y) for y in ordinary_datapoints_counts],
    showticklabels=False,
    row=1, col=2
)

# === Show or Save
fig.show()
fig.write_image("exp_v14_heatmap_accuracy_wide.pdf", format="pdf")

